# Differential Equations — Session 3  
## Section 1.3: Differential Equations as Mathematical Models

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture notebook

### Learning objectives

Students should be able to:

1. Translate verbal rate statements into differential equations.
2. Identify variables, parameters, units, assumptions, and initial data.
3. Distinguish model formulation from model solution.
4. Simulate and interpret growth/decay, cooling, epidemic, mixing, and falling-body models.
5. Explain how parameter changes alter qualitative behavior.

> The publisher section surveys many models. This 90-minute notebook develops five representative models computationally and gives shorter previews of the remaining applications.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Suggested pacing

| Time | Activity |
|---:|---|
| 0–12 min | Modeling cycle and units |
| 12–27 min | Exponential growth and decay |
| 27–42 min | Newton cooling/warming |
| 42–60 min | Disease spread |
| 60–73 min | Mixing tank |
| 73–84 min | Falling with air resistance |
| 84–90 min | Model gallery and exit check |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from IPython.display import display
from scipy.integrate import solve_ivp

try:
    from ipywidgets import interact, FloatSlider, IntSlider
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=4, suppress=True)

print("Notebook ready.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)

## 1. The modeling cycle

A useful workflow is:

1. **Define the system and goal.**
2. **Choose variables and units.**
3. **State assumptions.**
4. **Translate rate statements into equations.**
5. **Specify parameters and initial conditions.**
6. **Analyze or simulate.**
7. **Compare with reality and revise.**

A model is deliberately incomplete: it includes the mechanisms needed for the stated purpose.

### Classroom Checkpoint — Build the Balance Law

A tank contains an amount $A(t)$ of salt. Brine enters at rate $r_{\mathrm{in}}$ with concentration $c_{\mathrm{in}}$, and leaves at rate $r_{\mathrm{out}}$ with tank concentration $A(t)/V(t)$. Write the governing balance equation.

> Pause here. Before continuing, try to explain their reasoning before continuing.

In [ ]:
# Original modeling-cycle diagram
fig, ax = plt.subplots(figsize=(10, 4))
ax.axis("off")

steps = [
    "System\n& goal",
    "Variables\n& units",
    "Assumptions",
    "Differential\nequation",
    "Analysis /\nsimulation",
    "Validation\n& revision",
]
xpos = np.linspace(0.08, 0.92, len(steps))

for x, label in zip(xpos, steps):
    ax.text(x, 0.55, label, ha="center", va="center",
            bbox=dict(boxstyle="round,pad=0.5"))
for i in range(len(steps)-1):
    ax.annotate("", xy=(xpos[i+1]-0.055, 0.55),
                xytext=(xpos[i]+0.055, 0.55),
                arrowprops=dict(arrowstyle="->"))
ax.annotate("revise", xy=(xpos[1], 0.25), xytext=(xpos[-1], 0.25),
            ha="center", arrowprops=dict(arrowstyle="->",
            connectionstyle="arc3,rad=-0.25"))
ax.set_title("A differential-equation modeling cycle")
plt.show()

## 2. Exponential growth and decay

The statement

> The rate of change is proportional to the current amount

becomes

$$
\frac{dP}{dt}=kP.
$$

Its solution is

$$
P(t)=P_0e^{kt}.
$$

- $k>0$: growth
- $k<0$: decay
- Units of $k$: inverse time

In [ ]:
def exponential_demo(P0=100.0, k=0.25):
    t = np.linspace(0, 12, 500)
    P = P0*np.exp(k*t)
    plt.plot(t, P, linewidth=2)
    plt.scatter([0], [P0], s=70, label=f"P(0)={P0:g}")
    plt.xlabel("time")
    plt.ylabel("P(t)")
    plt.title(fr"$P'=kP$ with $k={k:.2f}$")
    plt.legend()
    plt.show()

    if k > 0:
        print(f"Doubling time = {np.log(2)/k:.3f} time units")
    elif k < 0:
        print(f"Half-life = {np.log(2)/abs(k):.3f} time units")
    else:
        print("The amount remains constant.")

if WIDGETS_AVAILABLE:
    interact(
        exponential_demo,
        P0=FloatSlider(min=20, max=300, step=10, value=100),
        k=FloatSlider(min=-0.5, max=0.5, step=0.05, value=0.25)
    )
else:
    exponential_demo()

### Modeling question

A bacterial culture increases by about $18\%$ per hour under ideal conditions.

A continuous-time model uses

$$
k=\ln(1.18)\approx0.1655\ \text{hour}^{-1},
$$

not $k=0.18$, because $e^k=1.18$ over one hour.

## 3. Newton's law of cooling or warming

Let $T(t)$ be an object's temperature and $T_m$ the ambient temperature.

The temperature-change rate is proportional to the temperature difference:

$$
\frac{dT}{dt}=-k(T-T_m),
\qquad k>0.
$$

The solution is

$$
T(t)=T_m+(T_0-T_m)e^{-kt}.
$$

The ambient temperature is an equilibrium.

In [ ]:
def cooling_demo(T0=90.0, Tm=22.0, k=0.18):
    t = np.linspace(0, 30, 500)
    T = Tm + (T0-Tm)*np.exp(-k*t)
    plt.plot(t, T, linewidth=2, label="object temperature")
    plt.axhline(Tm, linestyle="--", label="ambient temperature")
    plt.scatter([0], [T0], s=70)
    plt.xlabel("time (minutes)")
    plt.ylabel("temperature")
    plt.title("Newton cooling/warming")
    plt.legend()
    plt.show()

if WIDGETS_AVAILABLE:
    interact(
        cooling_demo,
        T0=FloatSlider(min=0, max=100, step=2, value=90),
        Tm=FloatSlider(min=0, max=50, step=1, value=22),
        k=FloatSlider(min=0.02, max=0.5, step=0.02, value=0.18)
    )
else:
    cooling_demo()

### Interpretation prompts

- What determines whether the object warms or cools?
- Can the model cross the ambient temperature?
- What does increasing $k$ do?
- Which assumption may fail if the surrounding temperature changes?

## 4. Spread of a disease: an SI model

Let:

- $S(t)$: susceptible people,
- $I(t)$: infected people,
- $N=S+I$: fixed population.

Assuming homogeneous mixing and no recovery,

$$
\frac{dI}{dt}=\beta\frac{SI}{N},
\qquad
\frac{dS}{dt}=-\beta\frac{SI}{N}.
$$

Because $S=N-I$,

$$
I'=\beta I\left(1-\frac{I}{N}\right),
$$

which is a logistic equation.

In [ ]:
def simulate_si(N=1000, I0=10, beta=0.45, days=30):
    def rhs(t, z):
        S, I = z
        new_infections = beta*S*I/N
        return [-new_infections, new_infections]

    t_eval = np.linspace(0, days, 500)
    sol = solve_ivp(rhs, (0, days), [N-I0, I0], t_eval=t_eval)

    plt.plot(sol.t, sol.y[0], label="susceptible S(t)")
    plt.plot(sol.t, sol.y[1], label="infected I(t)")
    plt.xlabel("time (days)")
    plt.ylabel("people")
    plt.title("SI disease-spread model")
    plt.legend()
    plt.show()

if WIDGETS_AVAILABLE:
    interact(
        simulate_si,
        N=IntSlider(min=200, max=2000, step=100, value=1000),
        I0=IntSlider(min=1, max=100, step=1, value=10),
        beta=FloatSlider(min=0.05, max=1.0, step=0.05, value=0.45),
        days=IntSlider(min=10, max=80, step=5, value=30)
    )
else:
    simulate_si()

### Assumptions worth challenging

- Everyone mixes uniformly.
- Population size is fixed.
- Infection is permanent.
- The contact parameter $\beta$ is constant.
- No births, deaths, vaccination, or behavior change occur.

The value of a model often lies in making assumptions explicit enough to test or improve.

## 5. A mixing-tank model

A tank contains $V$ liters of well-mixed liquid. Let $A(t)$ be the amount of salt in grams.

For equal inflow and outflow rates $r$,

$$
\frac{dA}{dt}
=
\underbrace{r\,c_{\text{in}}}_{\text{rate in}}
-
\underbrace{r\frac{A}{V}}_{\text{rate out}}.
$$

The units are grams per minute on both terms.

In [ ]:
def mixing_demo(A0=300.0, V=100.0, r=5.0, c_in=1.0, minutes=80):
    def rhs(t, A):
        return r*c_in - r*A[0]/V

    t_eval = np.linspace(0, minutes, 500)
    sol = solve_ivp(rhs, (0, minutes), [A0], t_eval=t_eval)

    equilibrium = V*c_in
    plt.plot(sol.t, sol.y[0], label="salt amount A(t)")
    plt.axhline(equilibrium, linestyle="--",
                label=f"equilibrium V c_in={equilibrium:g}")
    plt.xlabel("time (minutes)")
    plt.ylabel("salt (grams)")
    plt.title("Well-mixed tank with equal inflow and outflow")
    plt.legend()
    plt.show()

if WIDGETS_AVAILABLE:
    interact(
        mixing_demo,
        A0=FloatSlider(min=0, max=800, step=25, value=300),
        V=FloatSlider(min=50, max=200, step=10, value=100),
        r=FloatSlider(min=1, max=15, step=1, value=5),
        c_in=FloatSlider(min=0, max=5, step=0.25, value=1),
        minutes=IntSlider(min=20, max=160, step=10, value=80)
    )
else:
    mixing_demo()

## 6. Falling with linear air resistance

Take downward as positive. For an object of mass $m$,

- gravity contributes $mg$,
- linear drag contributes $-cv$.

Newton's law gives

$$
m\frac{dv}{dt}=mg-cv.
$$

The terminal velocity is obtained by setting $v'=0$:

$$
v_\infty=\frac{mg}{c}.
$$

In [ ]:
def falling_demo(m=70.0, c=14.0, v0=0.0):
    g = 9.81
    t = np.linspace(0, 20, 500)
    v_terminal = m*g/c
    v = v_terminal + (v0-v_terminal)*np.exp(-(c/m)*t)

    plt.plot(t, v, linewidth=2, label="velocity")
    plt.axhline(v_terminal, linestyle="--",
                label=f"terminal velocity={v_terminal:.2f} m/s")
    plt.scatter([0], [v0], s=70)
    plt.xlabel("time (s)")
    plt.ylabel("downward velocity (m/s)")
    plt.title("Falling body with linear air resistance")
    plt.legend()
    plt.show()

if WIDGETS_AVAILABLE:
    interact(
        falling_demo,
        m=FloatSlider(min=20, max=120, step=5, value=70),
        c=FloatSlider(min=2, max=40, step=2, value=14),
        v0=FloatSlider(min=-20, max=30, step=2, value=0)
    )
else:
    falling_demo()

## 7. Model gallery: important previews

The publisher section also introduces several models that will return later.

| Application | Typical state variable | Governing idea |
|---|---|---|
| Radioactive decay | amount $A(t)$ | $A'=-kA$ |
| Chemical reaction | product amount $X(t)$ | rate proportional to reactant concentrations |
| Draining tank | fluid depth $h(t)$ | Torricelli outflow proportional to $\sqrt h$ |
| RLC circuit | charge $q(t)$ | Kirchhoff voltage balance |
| Falling without drag | position $s(t)$ | $s''=-g$ with upward positive |
| Suspended cable | cable profile $y(x)$ | force balance along the cable |

The key skill today is not memorizing every model. It is recognizing the translation:

$$
\text{verbal rate law}
\longrightarrow
\text{units-consistent differential equation}.
$$

## Exit check

A medication leaves the bloodstream at a rate proportional to the amount present.

1. Define a state variable and units.
2. Write the differential equation.
3. State the sign of the proportionality constant.
4. Name one assumption.
5. Predict the long-term behavior.

A suitable model is $A'=-kA$, $k>0$, with $A(t)\to0$.

## Classroom Checkpoint — Closing Reflection

Before continuing, try to state the central method or theorem of this lesson, including its assumptions and one situation in which it is useful.

> Discuss first; run the next cell for an instructor summary.